In [ ]:
import csv
import requests
from io import StringIO
from google.colab import files

SHEET_ID = "118Znzcf8Nr2y0-YPCnO3JbZX-QZhm39q7ATDTpouVjE"
GID = ""  # gid of the Attachments tab (the number after #gid= in the tab's URL)

SHEET_URL = (
    f"https://docs.google.com/spreadsheets/d/"
    f"{SHEET_ID}/export?format=csv&gid={GID}"
)

response = requests.get(SHEET_URL)
response.raise_for_status()

reader = list(csv.DictReader(StringIO(response.content.decode("utf-8")), delimiter=","))


In [ ]:
ID = "id"
STATUS_COL = "#Status"
STATUS_ACTIVE = "Active"
STATUS_DISABLED = "Disabled"

# Column types. A new column not listed here falls back to guess_value.
STRING_KEYS = {"id", "name", "icon", "size", "family", "model", "page"}
NUMBER_KEYS = {"cost", "tier", "weight", "max_stock", "scrap"}
BOOL_KEYS = {"reserved", "any", "mag"}
# Cell "a, b, c" -> {"a", "b", "c"}
LIST_KEYS = {"comps", "guns", "calibers", "types"}
# Ready Lua expression, copied as is (e.g. {SteyrScout_1 = point(-20, 0, 0)})
RAW_KEYS = {"offsets"}

# Every header column except those starting with "#"; prefix a column with "#" to drop it.
header = reader[0].keys() if reader else []
KEYS = [k for k in header if not k.startswith("#")]


def lua_string(s):
    return '"' + s.replace("\\", "\\\\").replace('"', '\\"') + '"'


def is_number(s):
    return s.lstrip("-").replace(".", "", 1).isdigit()


def guess_value(value):
    if value in ("TRUE", "FALSE"):
        return value.lower()
    if is_number(value) or (value.startswith("{") and value.endswith("}")):
        return value
    return lua_string(value)


def format_value(key, raw_value, item_id):
    """Cell value -> Lua expression; None omits the field."""
    value = raw_value.strip()
    if key in BOOL_KEYS:
        if value.upper() == "TRUE":
            return "true"
        if value.upper() in ("FALSE", ""):
            return None
        raise ValueError(f'{item_id}.{key}: expected TRUE/FALSE, got "{value}"')
    if value == "":
        return None
    if key in STRING_KEYS:
        return lua_string(value)
    if key in NUMBER_KEYS:
        number = value.replace(",", "")
        if not is_number(number) or "." in number:
            raise ValueError(f'{item_id}.{key}: expected an integer, got "{value}"')
        return number
    if key in LIST_KEYS:
        elements = [e.strip() for e in value.split(",") if e.strip()]
        return "{" + ", ".join(lua_string(e) for e in elements) + "}"
    if key in RAW_KEYS:
        return value
    return guess_value(value)


def get_item_lua_snippet(row):
    fields = []
    for key in KEYS:
        lua_value = format_value(key, row.get(key) or "", row[ID])
        if lua_value is not None:
            fields.append(f"        {key} = {lua_value}")
    return "    {\n" + ",\n".join(fields) + "\n    }"


In [ ]:
LUA_FILE_NAME = "FEATURE_AttachmentItemsData.lua"

active_rows = []
ignored = 0
seen = set()

for row in reader:
    item_id = (row.get(ID) or "").strip()
    status = (row.get(STATUS_COL) or "").strip()
    if not item_id:
        continue
    if status == STATUS_DISABLED:
        ignored += 1
        continue
    if status != STATUS_ACTIVE:
        print(f'Warning: row "{item_id}" has unknown/empty #Status ("{status}"), skipped.')
        ignored += 1
        continue
    if item_id in seen:
        raise ValueError(f'duplicate id: "{item_id}"')
    seen.add(item_id)
    active_rows.append(row)

print(f"{len(active_rows)} attachments generated, {ignored} skipped (Disabled or unknown status).")

snippets = [get_item_lua_snippet(row) for row in active_rows]

with open(LUA_FILE_NAME, "w", encoding="utf-8", newline="\n") as lua_file:
    lua_file.write("-- ========== THIS IS AN AUTOMATICALLY GENERATED FILE! ==========\n")
    lua_file.write("-- Source: attachments sheet via GBO_AttachmentsToLua.ipynb. Edit the sheet, not this file.\n\n")
    lua_file.write("RAT_ATT_ITEMS = {\n")
    lua_file.write(",\n".join(snippets))
    lua_file.write("\n}\n")

print(snippets[0] if snippets else "")
files.download(LUA_FILE_NAME)
